# Instructor Notebook  --  Operation GRIDLOCK

---

| Phase | Time | Action |
|-------|------|--------|
| Reveal budget + show all designs | 10 min | Section 1 |
| Blue Team quick adjustment | 10 min | Students re-run BT cell |
| Red Team attack planning | 25 min | Students work in RT notebook |
| Live simulation (one matchup at a time) | 40 min | Section 2 |
| Debrief | 30 min | Section 3 |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as _tk
import numpy as np
import glob, os
# =============================================================
#  OPERATION GRIDLOCK -- Constants  (do not edit)
# =============================================================
import math as _math

BLUE_BUDGET = 500   # units available to Blue Teams

# ---- Relay nodes --------------------------------------------------------
#   All relay candidates are identical: 12u each to activate.
#   Relay nodes CANNOT be directly attacked -- only links can be attacked.
RELAY_NODE_COST = 12

# ---- Link specification -------------------------------------------------
#   Every link has TWO independent properties chosen by the Blue Team:
#
#   CAPACITY  1 | 2 | 3   (flow per direction, and build cost scale)
#     cap=1 : light wire      build = max(3,  round(dist * 2))
#     cap=2 : standard cable  build = max(6,  round(dist * 4))
#     cap=3 : heavy cable     build = max(10, round(dist * 6))
#
#   ARMORED  True | False   (immune to attack, costs 2.5x more to build)
#     armored=False : unprotected -- Red Team CAN attack this link
#     armored=True  : hardened    -- Red Team CANNOT attack this link
#                     build cost = base * 2.5  (rounded)
#
#   ATTACK COSTS  (flat -- independent of link length):
#   Longer unarmored lines have more physical exposure (more pylons, more
#   cable). The attacker only needs ONE weak point anywhere along the line,
#   so effort is roughly constant regardless of length.
#
#     sever   cap=1:  6u    sever   cap=2: 12u    sever   cap=3: 20u
#     degrade cap=1:  3u    degrade cap=2:  6u    degrade cap=3: 10u
#
#   Any unarmored link can be attacked, regardless of position:
#     source->relay, relay->relay, or relay->sink.
#   Only armored links are immune.

LINK_BUILD_RATE = {1: 2, 2: 4, 3: 6}    # build cost per distance unit
LINK_BUILD_MIN  = {1: 3, 2: 6, 3: 10}   # minimum build cost
ARMOR_MULT      = 2.5                    # armored cost multiplier

ATK_SEVER   = {1:  6, 2: 12, 3: 20}    # flat sever cost by cap
ATK_DEGRADE = {1:  3, 2:  6, 3: 10}    # flat degrade cost by cap

# ---- Attack budget uncertainty ------------------------------------------
RED_MIN = 55
RED_MAX = 100

# ---- Network topology  (fixed -- same for all Blue Teams) ---------------
#
#  DESIGN RULES
#  ============
#  1. Sources connect ONLY to relay nodes  (never directly to sinks).
#  2. Sinks  connect ONLY from relay nodes (never directly from sources).
#  3. No direct source->sink links.
#     Such a link would bypass the relay layer and, if armored, provide
#     completely unattackable supply -- trivially overpowered.
#
#  Total supply (21) > total demand (14).
#  A well-connected relay network achieves 100% demand satisfaction intact.

SOURCES = {
    "S1": {"x": 1, "y": 9, "supply": 8, "label": "NW Plant"},
    "S2": {"x": 9, "y": 9, "supply": 4, "label": "NE Plant"},
    "S3": {"x": 1, "y": 1, "supply": 5, "label": "SW Plant"},
    "S4": {"x": 9, "y": 1, "supply": 4, "label": "SE Plant"},
}

SINKS = {
    "D1": {"x": 2, "y": 8, "demand": 2, "label": "N-West"},
    "D2": {"x": 5, "y": 9, "demand": 1, "label": "N-Mid"},
    "D3": {"x": 8, "y": 8, "demand": 2, "label": "N-East"},
    "D4": {"x": 1, "y": 5, "demand": 1, "label": "W-Mid"},
    "D5": {"x": 5, "y": 5, "demand": 3, "label": "Downtown"},
    "D6": {"x": 9, "y": 5, "demand": 1, "label": "E-Mid"},
    "D7": {"x": 2, "y": 2, "demand": 1, "label": "S-West"},
    "D8": {"x": 5, "y": 1, "demand": 2, "label": "S-Mid"},
    "D9": {"x": 8, "y": 2, "demand": 1, "label": "S-East"},
}
TOTAL_DEMAND = sum(v["demand"] for v in SINKS.values())   # = 14

RELAY_CANDIDATES = {
    "RC1":  {"x": 3, "y": 8},
    "RC2":  {"x": 7, "y": 8},
    "RC3":  {"x": 2, "y": 6},
    "RC4":  {"x": 4, "y": 7},
    "RC5":  {"x": 6, "y": 7},
    "RC6":  {"x": 8, "y": 6},
    "RC7":  {"x": 3, "y": 5},
    "RC8":  {"x": 7, "y": 5},
    "RC9":  {"x": 2, "y": 3},
    "RC10": {"x": 5, "y": 3},
    "RC11": {"x": 4, "y": 5},
    "RC12": {"x": 7, "y": 3},
}

# ---- Position lookup (all node types) -----------------------------------
_POSITIONS = {}
_POSITIONS.update({k: (v["x"], v["y"]) for k, v in SOURCES.items()})
_POSITIONS.update({k: (v["x"], v["y"]) for k, v in SINKS.items()})
_POSITIONS.update({k: (v["x"], v["y"]) for k, v in RELAY_CANDIDATES.items()})


# ---- Cost functions -----------------------------------------------------
def _dist(n1, n2):
    x1, y1 = _POSITIONS[n1]
    x2, y2 = _POSITIONS[n2]
    return _math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

def link_build_cost(n1, n2, cap, armored):
    # Build cost: scales with distance and capacity; armored x2.5.
    d    = _dist(n1, n2)
    base = max(LINK_BUILD_MIN[cap], round(d * LINK_BUILD_RATE[cap]))
    return round(base * ARMOR_MULT) if armored else base


# ---- Visual palette -----------------------------------------------------
PAL = {
    "panel":        "#f8f9fa",
    "grid":         "#e9ecef",
    "border":       "#adb5bd",
    "text":         "#212529",
    "muted":        "#6c757d",
    "source":       "#2b8a3e",
    "sink":         "#1864ab",
    "relay":        "#e67700",
    "candidate":    "#ced4da",
    "relay_fill":   "#a5d8ff",
    "unprotected":  "#868e96",   # grey  -- attackable
    "armored":      "#2f9e44",   # green -- immune
    "dead":         "#c92a2a",
    "degraded":     "#f59f00",
    "ok":           "#2b8a3e",
    "warn":         "#e67700",
}
EDGE_W = {1: 1.4, 2: 2.8, 3: 4.5}   # line width by capacity

import matplotlib.pyplot as _plt
import matplotlib.patches as _mpa
import matplotlib.lines as _mlines
import matplotlib.patheffects as _pe


def draw_network(nodes_df, edges_df, ax=None, title="",
                 dead_edges=None, degraded_edges=None,
                 show_candidates=True):
    """
    Draw the network.
    Edge colour encodes protection:  grey = unprotected, green = armored.
    Edge width  encodes capacity:    thin=1, medium=2, thick=3.
    """
    dead_edges     = set(dead_edges    or [])
    degraded_edges = set(degraded_edges or [])
    standalone = ax is None
    if standalone:
        fig, ax = _plt.subplots(figsize=(11, 10), facecolor="white")

    ax.set_facecolor(PAL["panel"])
    ax.set_xlim(0.2, 10.8); ax.set_ylim(0.2, 10.8)
    ax.set_aspect("equal")
    ax.set_title(title, color=PAL["text"], fontsize=12,
                 fontweight="bold", pad=10, fontfamily="monospace")
    ax.tick_params(which="both", left=False, bottom=False,
                   labelleft=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_color(PAL["border"]); sp.set_linewidth(0.7)

    # Grid
    for v in range(1, 11):
        ax.axvline(v, color=PAL["grid"], lw=0.5, zorder=0)
        ax.axhline(v, color=PAL["grid"], lw=0.5, zorder=0)
        ax.text(v, 0.12, str(v), ha="center", fontsize=5.5, color=PAL["muted"])
        ax.text(0.12, v, str(v), va="center", fontsize=5.5, color=PAL["muted"])

    # Ghost relay candidates
    if show_candidates:
        for cid, cv in RELAY_CANDIDATES.items():
            ax.scatter(cv["x"], cv["y"], s=50, color=PAL["candidate"],
                       marker="s", zorder=1, alpha=0.55)
            ax.text(cv["x"] + 0.13, cv["y"] + 0.13, cid,
                    fontsize=5.5, color=PAL["candidate"], zorder=2)

    # Position lookup
    pos = {}
    for r in nodes_df.itertuples():
        pos[r.id] = (r.x, r.y)
    for nid, nd in SOURCES.items():
        pos[nid] = (nd["x"], nd["y"])
    for nid, nd in SINKS.items():
        pos[nid] = (nd["x"], nd["y"])

    # Draw edges
    for r in edges_df.itertuples():
        s, t = r.source, r.target
        if s not in pos or t not in pos:
            continue
        xs = [pos[s][0], pos[t][0]]
        ys = [pos[s][1], pos[t][1]]
        cap     = int(r.cap)
        armored = bool(r.armored)
        lw  = EDGE_W[cap]
        col = PAL["armored"] if armored else PAL["unprotected"]

        if r.id in dead_edges:
            ax.plot(xs, ys, color=PAL["dead"], lw=1.0, ls="--", alpha=0.28, zorder=2)
        elif r.id in degraded_edges:
            ax.plot(xs, ys, color=PAL["degraded"], lw=lw, ls=(0,(3,2)), alpha=0.75, zorder=2)
        else:
            ls = "-" if armored else (0, (5, 2))   # solid=armored, dashed=unprotected
            ax.plot(xs, ys, color=col, lw=lw, ls=ls, alpha=0.80, zorder=2)
            # Build cost label at midpoint
            mx = (pos[s][0] + pos[t][0]) / 2
            my = (pos[s][1] + pos[t][1]) / 2
            try:
                bc = link_build_cost(s, t, cap, armored)
                ax.text(mx, my, str(bc) + "u",
                        ha="center", va="center",
                        fontsize=4.5, color=col, alpha=0.80,
                        bbox=dict(fc="white", ec="none", pad=0.5), zorder=3)
            except Exception:
                pass

    # Source nodes
    for nid, nd in SOURCES.items():
        ax.scatter(nd["x"], nd["y"], s=360, color=PAL["source"],
                   marker="^", edgecolors="white", linewidths=2.0, zorder=6)
        ax.annotate(nid + " (" + str(nd["supply"]) + ")",
                    (nd["x"], nd["y"]),
                    textcoords="offset points", xytext=(7, 6),
                    fontsize=7.5, fontweight="bold", color=PAL["source"], zorder=7,
                    path_effects=[_pe.withStroke(linewidth=2, foreground="white")])

    # Sink nodes (larger circle = more demand)
    for nid, nd in SINKS.items():
        sz  = 120 + nd["demand"] * 55
        ax.scatter(nd["x"], nd["y"], s=sz, color=PAL["sink"],
                   marker="o", edgecolors="white", linewidths=1.5, zorder=5)
        ax.annotate(nid + " (" + str(nd["demand"]) + ")",
                    (nd["x"], nd["y"]),
                    textcoords="offset points", xytext=(5, 5),
                    fontsize=6.5, fontweight="bold", color=PAL["sink"], zorder=6,
                    path_effects=[_pe.withStroke(linewidth=1.5, foreground="white")])

    # Relay nodes
    for r in nodes_df.itertuples():
        if r.role != "relay":
            continue
        ax.scatter(r.x, r.y, s=220, color=PAL["relay_fill"],
                   marker="s", edgecolors=PAL["relay"],
                   linewidths=2.0, zorder=4)
        ax.annotate(r.id, (r.x, r.y),
                    textcoords="offset points", xytext=(5, 6),
                    fontsize=7, fontweight="bold", color=PAL["text"], zorder=5,
                    path_effects=[_pe.withStroke(linewidth=2, foreground="white")])

    # Legend -- upper right
    items = [
        _mpa.Patch(color=PAL["source"],     label="Source  (supply in brackets)"),
        _mpa.Patch(color=PAL["sink"],       label="Sink  (demand in brackets)"),
        _mpa.Patch(color=PAL["relay_fill"], label="Relay node  (12u each)"),
        _mlines.Line2D([], [], color=PAL["unprotected"], lw=2.4, ls=(0,(5,2)),
                       label="Unprotected link  (attackable)"),
        _mlines.Line2D([], [], color=PAL["armored"],     lw=2.4, ls="-",
                       label="Armored link  (immune, ×2.5 cost)"),
        _mlines.Line2D([], [], color=PAL["unprotected"], lw=EDGE_W[1], ls="-",
                       label="Cap=1  (sever 6u / degrade 3u)"),
        _mlines.Line2D([], [], color=PAL["unprotected"], lw=EDGE_W[2], ls="-",
                       label="Cap=2  (sever 12u / degrade 6u)"),
        _mlines.Line2D([], [], color=PAL["unprotected"], lw=EDGE_W[3], ls="-",
                       label="Cap=3  (sever 20u / degrade 10u)"),
    ]
    ax.legend(handles=items, loc="upper right",
              framealpha=0.95, facecolor="white",
              edgecolor=PAL["border"], labelcolor=PAL["text"], fontsize=7.5)

    if standalone:
        _plt.tight_layout()
        _plt.show()


def draw_budget_bar(spent, budget, breakdown, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = _plt.subplots(figsize=(11, 1.4), facecolor="white")
    ax.set_facecolor(PAL["panel"])
    ax.set_xlim(0, budget); ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Build units", color=PAL["muted"], fontsize=8)
    ax.set_title("Budget", color=PAL["text"], fontsize=10,
                 fontweight="bold", fontfamily="monospace")
    for sp in ax.spines.values():
        sp.set_color(PAL["border"]); sp.set_linewidth(0.7)
    ax.tick_params(colors=PAL["muted"])
    x = 0
    for lbl, val, col in breakdown:
        if val <= 0:
            continue
        ax.barh(0.5, val, left=x, height=0.48, color=col, alpha=0.88)
        if val > 15:
            ax.text(x + val / 2, 0.5, lbl + "  " + str(val) + "u",
                    ha="center", va="center",
                    color="white", fontsize=6.5, fontweight="bold")
        x += val
    remaining = budget - spent
    if remaining > 0:
        ax.barh(0.5, remaining, left=x, height=0.48,
                color=PAL["grid"], alpha=0.7)
    ax.axvline(budget, color=PAL["dead"], lw=1.5, ls="--")
    col = PAL["dead"] if remaining < 0 else PAL["ok"]
    ax.text(budget - 2, 0.88,
            str(spent) + "/" + str(budget) + "u  ("
            + str(remaining) + " remaining)",
            ha="right", color=col, fontsize=9, fontweight="bold")
    if standalone:
        _plt.tight_layout()


def draw_curve(curve, ax=None, label="", pred=None):
    standalone = ax is None
    if standalone:
        fig, ax = _plt.subplots(figsize=(8, 5), facecolor="white")
    import matplotlib.ticker as _tk
    ax.set_facecolor(PAL["panel"])
    for sp in ax.spines.values():
        sp.set_color(PAL["border"]); sp.set_linewidth(0.8)
    ax.tick_params(colors=PAL["muted"])
    ax.yaxis.set_major_formatter(
        _tk.FuncFormatter(lambda v, _: str(round(v * 100)) + "%"))
    ax.axhline(0.8, ls="--", color=PAL["warn"], lw=1.0, alpha=0.6)
    ax.text(0.01, 0.82, "80% target",
            transform=ax.get_yaxis_transform(), color=PAL["warn"], fontsize=8)
    steps = list(range(len(curve)))
    ax.fill_between(steps, curve, alpha=0.12, color=PAL["relay"])
    ax.plot(steps, curve, color=PAL["relay"], lw=2.2,
            marker="o", markersize=6,
            markerfacecolor="white", markeredgecolor=PAL["relay"],
            markeredgewidth=1.8, label=label)
    if pred is not None:
        ax.axhline(pred, ls=":", color=PAL["dead"], lw=1.2, alpha=0.7,
                   label="RT prediction " + str(round(pred * 100)) + "%")
    ax.set_ylim(-0.06, 1.15)
    ax.set_xlim(-0.3, max(len(curve) - 0.5, 1))
    ax.set_xlabel("Attack step", color=PAL["muted"])
    ax.set_ylabel("Demand fraction served", color=PAL["muted"])
    ax.set_title("Resilience Curve", color=PAL["text"], fontsize=11,
                 fontweight="bold", fontfamily="monospace")
    sc = score_resilience(curve)
    ax.set_title("Resilience Curve  |  R1=" + str(sc["R1"])
                 + "  R2=" + str(sc["R2"])
                 + "  Score=" + str(sc["total"]),
                 color=PAL["text"], fontsize=10,
                 fontweight="bold", fontfamily="monospace")
    ax.legend(framealpha=0.9, facecolor="white",
              edgecolor=PAL["border"], labelcolor=PAL["text"], fontsize=9)
    if standalone:
        _plt.tight_layout()
        _plt.show()

import networkx as nx


def _build_flow_graph(nodes_df, edges_df, dead_e, deg_e):
    """
    Build directed flow graph from current network state.

    edges_df columns: id, source, target, cap, armored
      cap    : int  1 | 2 | 3    -- flow capacity per direction
      armored: bool               -- True means immune to attack

    dead_e : set of edge ids fully severed
    deg_e  : set of edge ids degraded (capacity halved)
    """
    relay_ids = set(nodes_df["id"])
    all_nodes = relay_ids | set(SOURCES) | set(SINKS)

    DG = nx.DiGraph()
    DG.add_nodes_from(all_nodes)

    for r in edges_df.itertuples():
        if r.id in dead_e:
            continue
        s, t = r.source, r.target
        if s not in all_nodes or t not in all_nodes:
            continue
        cap = r.cap * (0.5 if r.id in deg_e else 1.0)
        DG.add_edge(s, t, capacity=cap)
        DG.add_edge(t, s, capacity=cap)

    DG.add_node("__SRC__")
    DG.add_node("__SNK__")
    for nid, nd in SOURCES.items():
        DG.add_edge("__SRC__", nid, capacity=nd["supply"])
    for nid, nd in SINKS.items():
        DG.add_edge(nid, "__SNK__", capacity=nd["demand"])

    return DG


def compute_perf(nodes_df, edges_df, dead_e, deg_e):
    """Return max-flow performance metrics."""
    DG = _build_flow_graph(nodes_df, edges_df, dead_e, deg_e)
    try:
        flow_val, _ = nx.maximum_flow(DG, "__SRC__", "__SNK__")
    except Exception:
        flow_val = 0.0
    fraction = round(min(flow_val / TOTAL_DEMAND, 1.0), 4)
    isolated = [nid for nid in SINKS
                if not nx.has_path(DG, "__SRC__", nid)]
    return {"fraction": fraction, "isolated": isolated,
            "flow": round(flow_val, 2)}


def run_attacks(nodes_df, edges_df, atk_df):
    """
    Execute the Red Team attack sequence.

    Attack rules:
      - Any unprotected (armored=False) link is a valid target.
        This includes source→relay, relay→relay, and relay→sink links.
      - Armored (armored=True) links cannot be attacked.
      - Relay nodes cannot be targeted.

    Actions:
      "sever"  : permanently removes the link.
                 cost = ATK_SEVER[cap]  (flat, independent of length)
      "degrade": halves the link's capacity.
                 cost = ATK_DEGRADE[cap]  (flat, independent of length)
                 A degraded link can still be severed for the sever cost.

    Returns: curve, events, dead_e, deg_e
    """
    relay_ids = set(nodes_df["id"])
    emap = {r.id: r for r in edges_df.itertuples()}

    # Attackable = any link where armored is False
    attackable = {r.id for r in edges_df.itertuples()
                  if not bool(r.armored)}

    dead_e = set()
    deg_e  = set()

    p0    = compute_perf(nodes_df, edges_df, dead_e, deg_e)
    curve = [p0["fraction"]]
    events = []

    for row in atk_df.sort_values("step").itertuples():
        tid    = str(row.target_id)
        action = str(row.action)
        ev     = {"step": row.step, "summary": "", "cost": 0}

        if str(getattr(row, "target_type", "edge")) == "node":
            ev["summary"] = "INVALID: relay nodes cannot be attacked. Target links."

        elif tid not in emap:
            ev["summary"] = "INVALID: unknown link id '" + tid + "'"

        elif bool(emap[tid].armored):
            ev["summary"] = ("INVALID: " + tid
                             + " is armored and cannot be attacked.")

        elif tid in dead_e:
            ev["summary"] = "INVALID: " + tid + " is already severed."

        elif action == "sever":
            cap  = int(emap[tid].cap)
            cost = ATK_SEVER[cap]
            dead_e.add(tid)
            deg_e.discard(tid)
            ev["cost"]    = cost
            ev["summary"] = ("Severed " + tid
                             + " (cap=" + str(cap) + ")"
                             + (" [was degraded]" if tid in deg_e else "")
                             + " -- " + str(cost) + "u")

        elif action == "degrade":
            if tid in deg_e:
                ev["summary"] = ("INVALID: " + tid
                                 + " already degraded. Use 'sever' to finish.")
            else:
                cap  = int(emap[tid].cap)
                cost = ATK_DEGRADE[cap]
                deg_e.add(tid)
                ev["cost"]    = cost
                ev["summary"] = ("Degraded " + tid
                                 + " (cap " + str(cap)
                                 + " → " + str(cap * 0.5) + ")"
                                 + " -- " + str(cost) + "u")
        else:
            ev["summary"] = ("INVALID action '" + action
                             + "'. Use 'sever' or 'degrade'.")

        p = compute_perf(nodes_df, edges_df, dead_e, deg_e)
        ev["frac"]     = p["fraction"]
        ev["isolated"] = p["isolated"]
        curve.append(p["fraction"])
        events.append(ev)

    return curve, events, dead_e, deg_e


def score_resilience(curve):
    R1 = round(sum(curve) / len(curve) * 100, 1)   # average throughput
    R2 = round(min(curve) * 100, 1)                 # worst-case floor
    return {"R1": R1, "R2": R2, "total": round(0.6 * R1 + 0.4 * R2, 1)}

import pandas as pd
import ast


def load_network_csv(path):
    """Load a Blue Team network CSV. Returns nodes_df, edges_df, meta dict."""
    df = pd.read_csv(path, dtype=str).fillna("")
    meta = {r["id"]: r["value"]
            for _, r in df[df["record"] == "META"].iterrows()}
    nodes = df[df["record"] == "NODE"].copy()
    edges = df[df["record"] == "EDGE"].copy()
    nodes_df = pd.DataFrame({
        "id":   nodes["id"].values,
        "role": nodes["role"].values,
        "x":    nodes["x"].astype(float).values,
        "y":    nodes["y"].astype(float).values,
        "label": nodes["label"].values,
    })
    # cap stored as int, armored stored as bool
    edges_df = pd.DataFrame({
        "id":      edges["id"].values,
        "source":  edges["source"].values,
        "target":  edges["target"].values,
        "cap":     edges["cap"].astype(int).values,
        "armored": edges["armored"].map(
                       lambda v: str(v).strip().lower() in ("true","1","yes")
                   ).values,
    })
    return nodes_df, edges_df, meta


def load_attack_csv(path):
    """Load a Red Team attack CSV. Returns atk_df, meta dict."""
    df = pd.read_csv(path, dtype=str).fillna("")
    meta = {r["id"]: r["value"]
            for _, r in df[df["record"] == "META"].iterrows()}
    atk = df[df["record"] == "ATTACK"].copy()
    atk_df = pd.DataFrame({
        "step":        atk["step"].astype(int).values,
        "target_type": atk["target_type"].values,
        "target_id":   atk["target_id"].values,
        "action":      atk["action"].values,
        "rationale":   atk["rationale"].values,
    })
    return atk_df, meta

print("Instructor notebook ready.")


---
## Section 1 -- Reveal Budget & Show Designs

In [ ]:
RED_BUDGET_REVEALED = 80   # <- set this, then project

print("=" * 56)
print("  OPERATION GRIDLOCK  --  ATTACK BUDGET REVEALED")
print()
print("  Red Team budget:  " + str(RED_BUDGET_REVEALED) + " units")
print("  Range shown to Blue Teams:  " + str(RED_MIN) + "u -- " + str(RED_MAX) + "u")
print()
print("  Attack costs (flat):")
print("  cap | sever | degrade")
for c in [1,2,3]:
    print("   " + str(c) + "  |  " + str(ATK_SEVER[c]) + "u  |  " + str(ATK_DEGRADE[c]) + "u")
print()
print("  Any unprotected link can be attacked.")
print("  Blue Teams: 10 min quick adjustment.")
print("  Red Teams:  25 min attack planning.")
print("=" * 56)


In [ ]:
NETWORK_FILES = sorted(glob.glob("BT*_network.csv"))
print("Found", len(NETWORK_FILES), "Blue Team files")
n = len(NETWORK_FILES)
if n == 0:
    print("No files found -- place BT*_network.csv here.")
else:
    cols_n = min(n, 3)
    rows_n = (n + cols_n - 1) // cols_n
    fig, axes = plt.subplots(rows_n, cols_n,
                              figsize=(11*cols_n, 10*rows_n),
                              facecolor="white")
    ax_list = list(axes.flat) if hasattr(axes, "flat") else [axes]
    for i, fpath in enumerate(NETWORK_FILES):
        nd, ed, mt = load_network_csv(fpath)
        nc = len(nd) * RELAY_NODE_COST
        ec = sum(link_build_cost(r.source, r.target, int(r.cap), bool(r.armored))
                 for r in ed.itertuples())
        n_arm = ed["armored"].sum()
        title = (mt.get("team_id","?") + " -- " + mt.get("team_name","?")
                 + "  |  " + str(nc+ec) + "/" + str(BLUE_BUDGET) + "u"
                 + "  " + str(n_arm) + " armored  " + mt.get("strategy","?"))
        draw_network(nd, ed, ax=ax_list[i], title=title, show_candidates=False)
    for j in range(n, rows_n*cols_n):
        ax_list[j].set_visible(False)
    plt.suptitle("OPERATION GRIDLOCK -- All Blue Team Designs",
                 color=PAL["text"], fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("all_designs.png", dpi=120,
                bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved -> all_designs.png")


---
## Section 2 -- Live Matchups

Change `SIM_NETWORK` and `SIM_ATTACK` for each pair.

In [ ]:
SIM_NETWORK = "BT1_network.csv"
SIM_ATTACK  = "RT1_vs_BT1_attacks.csv"

nodes_df, edges_df, net_meta = load_network_csv(SIM_NETWORK)
atk_df,   atk_meta           = load_attack_csv(SIM_ATTACK)
BT_ID = net_meta.get("team_id","?")
RT_ID = atk_meta.get("red_team_id","?")
PRED  = float(atk_meta.get("predicted_min","0"))

n_arm = edges_df["armored"].sum()
n_unp = len(edges_df) - n_arm
print("=" * 58)
print("  " + RT_ID + "  ->  " + BT_ID + " -- " + net_meta.get("team_name","?"))
print("  Strategy:  " + net_meta.get("strategy","?"))
print("  Plan:      " + atk_meta.get("chosen_plan","?"))
print("  Links:     " + str(len(edges_df))
      + "  (" + str(n_unp) + " unprotected, " + str(n_arm) + " armored)")
print("  RT prediction: min = " + str(round(PRED*100)) + "%")
print("=" * 58)

curve, events, dead_e, deg_e = run_attacks(nodes_df, edges_df, atk_df)
sc = score_resilience(curve)
print()
total_spent = 0
for ev in events:
    total_spent += ev["cost"]
    print("  Step " + str(ev["step"]) + ": " + ev["summary"])
    iso = ("  Isolated: " + str(ev["isolated"])) if ev["isolated"] else ""
    print("    -> " + str(round(ev["frac"]*100)) + "%" + iso)
print()
print("  Budget spent: " + str(total_spent) + "u")
print("  Score: R1=" + str(sc["R1"])
      + "  R2=" + str(sc["R2"])
      + "  Total=" + str(sc["total"]) + "/100")


In [ ]:
fig = plt.figure(figsize=(20,10), facecolor="white")
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05,
                         left=0.02, right=0.98, top=0.92, bottom=0.08)
ax_net   = fig.add_subplot(gs[0])
ax_curve = fig.add_subplot(gs[1])
draw_network(nodes_df, edges_df, ax=ax_net,
             title=BT_ID + " -- post-attack state",
             dead_edges=dead_e, degraded_edges=deg_e,
             show_candidates=False)
draw_curve(curve, ax_curve, label="Resilience -- " + BT_ID, pred=PRED)
plt.suptitle("OPERATION GRIDLOCK  |  " + RT_ID + "  ->  " + BT_ID,
             color=PAL["text"], fontsize=15, fontweight="bold")
out_f = BT_ID + "_vs_" + RT_ID + "_result.png"
plt.savefig(out_f, dpi=130, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved ->", out_f)


---
## Section 3 -- Debrief

In [ ]:
ALL_SCORES = {}
for atk_path in sorted(glob.glob("RT*_vs_BT*_attacks.csv")):
    atk_df, atk_meta = load_attack_csv(atk_path)
    bt_id    = atk_meta.get("target_team_id","?")
    net_path = bt_id + "_network.csv"
    if not os.path.exists(net_path):
        print("WARNING: missing", net_path); continue
    nd, ed, nm = load_network_csv(net_path)
    curve, *_  = run_attacks(nd, ed, atk_df)
    sc         = score_resilience(curve)
    ALL_SCORES[bt_id] = {"team_name": nm.get("team_name","?"),
                          "strategy":  nm.get("strategy","?"),
                          "rationale": nm.get("rationale",""),
                          "curve": curve, **sc}
    print(bt_id.ljust(6) + nm.get("team_name","?").ljust(24)
          + " R1=" + str(sc["R1"]).rjust(5)
          + "  R2=" + str(sc["R2"]).rjust(5)
          + "  Score=" + str(sc["total"]).rjust(6))


### Leaderboard

In [ ]:
if ALL_SCORES:
    teams = sorted(ALL_SCORES, key=lambda t: -ALL_SCORES[t]["total"])
    x, w  = np.arange(len(teams)), 0.35
    fig, ax = plt.subplots(figsize=(12,5), facecolor="white")
    ax.set_facecolor(PAL["panel"])
    for sp in ax.spines.values(): sp.set_color(PAL["border"]); sp.set_linewidth(0.8)
    ax.tick_params(colors=PAL["muted"])
    b1 = ax.bar(x-w/2, [ALL_SCORES[t]["R1"] for t in teams], w,
                label="R1 Absorption (60%)", color="#339af0", alpha=0.88,
                edgecolor="white", linewidth=0.8)
    b2 = ax.bar(x+w/2, [ALL_SCORES[t]["R2"] for t in teams], w,
                label="R2 Floor (40%)", color="#f76707", alpha=0.88,
                edgecolor="white", linewidth=0.8)
    for bars in [b1,b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.5, str(round(h)),
                    ha="center", va="bottom",
                    color=PAL["text"], fontsize=9, fontweight="bold")
    ax.set_ylim(0,115); ax.set_xticks(x)
    ax.set_xticklabels([t+"\n"+ALL_SCORES[t]["team_name"] for t in teams],
                       color=PAL["text"], fontsize=9)
    ax.set_ylabel("Score (0-100)", color=PAL["muted"])
    ax.axhline(80, color=PAL["muted"], lw=0.8, ls="--", alpha=0.5)
    ax.set_title("OPERATION GRIDLOCK -- Leaderboard",
                 color=PAL["text"], fontsize=14,
                 fontweight="bold", fontfamily="monospace")
    ax.legend(framealpha=0.9, facecolor="white",
              edgecolor=PAL["border"], labelcolor=PAL["text"], fontsize=10)
    plt.tight_layout()
    plt.savefig("leaderboard.png", dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()


### All Resilience Curves

In [ ]:
if ALL_SCORES:
    col_cycle = ["#1971c2","#2b8a3e","#e67700","#862e9c","#c92a2a","#0b7285"]
    teams = sorted(ALL_SCORES, key=lambda t: -ALL_SCORES[t]["total"])
    maxl  = max(len(ALL_SCORES[t]["curve"]) for t in teams)
    fig, ax = plt.subplots(figsize=(13,6), facecolor="white")
    ax.set_facecolor(PAL["panel"])
    for sp in ax.spines.values(): sp.set_color(PAL["border"]); sp.set_linewidth(0.8)
    ax.tick_params(colors=PAL["muted"])
    ax.axhline(0.80, ls="--", color=PAL["warn"], lw=1.0, alpha=0.7)
    ax.text(0.01, 0.82, "80% target",
            transform=ax.get_yaxis_transform(), color=PAL["warn"], fontsize=8)
    import matplotlib.ticker as _tk2
    ax.yaxis.set_major_formatter(
        _tk2.FuncFormatter(lambda v,_: str(round(v*100))+"%"))
    for i, tid in enumerate(teams):
        sc  = ALL_SCORES[tid]; c = sc["curve"]; col = col_cycle[i%len(col_cycle)]
        xv  = list(range(len(c)))
        ax.plot(xv, c, color=col, lw=2.0, marker="o", markersize=5,
                markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.5,
                label=tid+" -- "+sc["team_name"]+"  ("+str(sc["total"])+"pts)")
        ax.annotate(tid, (xv[-1], c[-1]),
                    textcoords="offset points", xytext=(4,0),
                    color=col, fontsize=8, fontweight="bold")
    ax.set_ylim(-0.06,1.15); ax.set_xlim(-0.3, maxl-0.5)
    ax.set_xlabel("Attack step", color=PAL["muted"])
    ax.set_ylabel("Demand fraction served", color=PAL["muted"])
    ax.set_title("All Resilience Curves",
                 color=PAL["text"], fontsize=12,
                 fontweight="bold", fontfamily="monospace")
    ax.legend(framealpha=0.92, facecolor="white",
              edgecolor=PAL["border"], labelcolor=PAL["text"], fontsize=9)
    plt.tight_layout()
    plt.savefig("all_curves.png", dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()


### Design Rationale Reveal

In [ ]:
for f in sorted(glob.glob("BT*_network.csv")):
    nd, ed, mt = load_network_csv(f)
    tid = mt.get("team_id","?")
    sc  = ALL_SCORES.get(tid,{})
    n_arm = ed["armored"].sum()
    nc    = len(nd)*RELAY_NODE_COST
    ec    = sum(link_build_cost(r.source,r.target,int(r.cap),bool(r.armored))
                for r in ed.itertuples())
    print("=" * 60)
    print("  " + tid + " -- " + mt.get("team_name","?"))
    print("  Strategy:  " + mt.get("strategy","?"))
    print("  Budget:    " + str(nc+ec) + "/" + str(BLUE_BUDGET) + "u")
    print("  Armored:   " + str(n_arm) + " / " + str(len(ed)) + " links")
    if sc:
        print("  Score:     R1=" + str(sc.get("R1","?"))
              + "  R2=" + str(sc.get("R2","?"))
              + "  Total=" + str(sc.get("total","?")) + "/100")
    print()
    print("  " + mt.get("rationale","(not provided)"))
    print()
